### Models:
1. KNN: content-based recommender (baseline)
2. LightGBM LambdaMART: learning-to-rank (main model)

In [3]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import lightgbm as lgb
import time

from datetime import datetime
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.metrics import ndcg_score

timestr = time.strftime("%Y%m%d-%H%M%S")

In [4]:
# plotting style 
plt.rcParams.update({
    "figure.facecolor":  "white",
    "axes.facecolor":    "white",
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "axes.grid":         True,
    "grid.color":        "#e4e4e4",
    "grid.linewidth":    0.5,
    "font.size":         10,
    "axes.titlesize":    11,
    "axes.titleweight":  "bold",
})

BLUE  = "#378ADD"
TEAL  = "#1D9E75"
AMBER = "#EF9F27"
CORAL = "#D85A30"
GRAY  = "#888780"

os.makedirs('../models',  exist_ok=True)
os.makedirs('../output',  exist_ok=True)
os.makedirs('../figures', exist_ok=True)

### Load Preprocessed Data

In [5]:
train = pd.read_csv('C:\\Users\\annad\\Documents\\UvAnna x VU\\Data Mining\\data_mining\\A2\\data\\processed\\train.csv')
val   = pd.read_csv('C:\\Users\\annad\\Documents\\UvAnna x VU\\Data Mining\\data_mining\\A2\\data\\processed\\val.csv')
test  = pd.read_csv('C:\\Users\\annad\\Documents\\UvAnna x VU\\Data Mining\\data_mining\\A2\\data\\processed\\test.csv')

print(f"Train: {len(train):,} rows | {train['srch_id'].nunique():,} searches")
print(f"Val:   {len(val):,} rows   | {val['srch_id'].nunique():,} searches")
print(f"Test:  {len(test):,} rows  | {test['srch_id'].nunique():,} searches")

Train: 3,964,501 rows | 159,836 searches
Val:   993,846 rows   | 39,959 searches
Test:  4,959,183 rows  | 199,549 searches


In [6]:
# Evaluation helper
def eval_ndcg5(df, score_col='score', relevance_col='relevance'):
    """Compute mean NDCG@5 across all searches."""
    scores = []
    for _, grp in df.groupby('srch_id'):
        true_rel = grp[relevance_col].values
        pred_rel = grp[score_col].values
        if len(true_rel) > 1 and true_rel.sum() > 0:
            scores.append(ndcg_score([true_rel], [pred_rel], k=5))
    return np.mean(scores)

### Model 1: KNN Content-Based Recommender
KNN is used as a content-based recommender: hotels with similar features to historically booked hotels score higher. 

https://www.geeksforgeeks.org/machine-learning/recommender-systems-using-knn/


In [ ]:
KNN_FEATURES = [
    'prop_starrating', 'prop_review_score',
    'prop_location_score1', 'prop_location_score2',
    'price_usd', 'promotion_flag',
    'srch_length_of_stay', 'srch_booking_window',
    'srch_adults_count', 'srch_children_count',
    'visitor_hist_starrating', 'visitor_hist_adr_usd',
    'new_customer', 
]

# sample by srch_id to keep CV feasible (preserves group structure)
sample_ids  = np.random.choice(train['srch_id'].unique(),
                                size=10000, replace=False)
df_sample   = train[train['srch_id'].isin(sample_ids)].copy()

X_knn       = df_sample[KNN_FEATURES]
y_knn       = df_sample['booking_bool']
groups_knn  = df_sample['srch_id']

In [8]:
# Cross-validated hyperparameter search
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn',    KNeighborsClassifier()),
])

param_grid = {
    'knn__n_neighbors': [10, 20, 50],
    'knn__weights':     ['distance'],
    'knn__metric':      [ 'manhattan'],
}

# GroupKFold keeps entire searches together, never splits a srch_id
cv = GroupKFold(n_splits=5)

grid_search = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring='average_precision',  # proxy for ranking quality
    n_jobs=-1,
    verbose=1,
)

grid_search.fit(X_knn, y_knn, groups=groups_knn)

print(f"Best params: {grid_search.best_params_}")
print(f"Best CV score: {grid_search.best_score_:.4f}")

Fitting 5 folds for each of 3 candidates, totalling 15 fits
Best params: {'knn__metric': 'manhattan', 'knn__n_neighbors': 50, 'knn__weights': 'distance'}
Best CV score: 0.0406


In [9]:
# Fit final and best KNN on full training set
best_knn = grid_search.best_estimator_
best_knn.fit(train[KNN_FEATURES], train['booking_bool'])

Pipeline(steps=[('scaler', StandardScaler()),
                ('knn',
                 KNeighborsClassifier(metric='manhattan', n_neighbors=50,
                                      weights='distance'))])

In [ ]:
# evaluate on validation set
val['knn_score'] = best_knn.predict_proba(val[KNN_FEATURES])[:, 1]
knn_ndcg = eval_ndcg5(val, score_col='knn_score')
print(f"KNN NDCG@5 (validation): {knn_ndcg:.4f}")

In [ ]:
# save KNN model
date_str = datetime.now().strftime("%Y%m%d")
joblib.dump(best_knn, f'../models/knn_{date_str}.pkl')
print(f"KNN saved: ../models/knn_{date_str}.pkl")

In [10]:
# evaluate on test set and save submission
test['knn_score'] = best_knn.predict_proba(test[KNN_FEATURES])[:, 1]

submission_knn = (test
                  .sort_values(['srch_id', 'knn_score'],
                               ascending=[True, False])
                  [['srch_id', 'prop_id']]
                  .rename(columns={'srch_id': 'SearchId',
                                   'prop_id': 'PropertyId'}))

submission_knn.to_csv(
    f'../output/VU-DM-2026-Group-62-knn-{timestr}.csv', index=False)
print(f"KNN submission saved ({len(submission_knn):,} rows)")

KeyError: "['new_customer'] not in index"

### Model 2: LightGBM LambdaMART
LambdaMART directly optimises a ranking objective (NDCG). Unlike pointwise classifiers, it learns from pairs of hotels within the same search, weighted by the NDCG gain of reordering them. The `group` parameter tells LightGBM which rows belong to the same query.

In [ ]:
LGB_FEATURES = [
    # hotel features
    'prop_starrating', 'prop_review_score', 'prop_brand_bool',
    'prop_location_score1', 'prop_location_score2',
    'prop_log_historical_price', 'price_usd', 'promotion_flag',

    # search features
    'srch_length_of_stay', 'srch_booking_window',
    'srch_adults_count', 'srch_children_count',
    'srch_room_count', 'srch_saturday_night_bool',

    # user features
    'visitor_hist_starrating', 'visitor_hist_adr_usd', 'new_customer',

    # within-search rank features (most impactful)
    'price_rank', 'price_ratio', 'price_rank_pct',
    'starrating_rank', 'review_rank', 'loc1_rank', 'loc2_rank',

    # quality and interaction features
    'quality_score', 'ad_vs_real', 'location_score_combined',
    'star_vs_hist', 'price_vs_hist', 'bool_same_country',

    # hotel historical performance
    'prop_booking_rate', 'prop_click_rate',
    'prop_mean_price', 'prop_std_price', 'prop_n_appearances',

    # prop-destination
    'prop_dest_booking_rate', 'prop_dest_click_rate',

    # estimated position (from non-random sort in training)
    'estimated_position',

    # competitor features
    'is_cheapest_any', 'is_expensive_any',
    'comp_score_sum', 'comp_advantage',

    # temporal
    'month', 'day_of_week', 'hour',
    'is_last_minute', 'is_long_stay',
]

# keep only columns that exist in the dataframe
LGB_FEATURES = [f for f in LGB_FEATURES if f in train.columns]
print(f"Using {len(LGB_FEATURES)} features")

https://lightgbm.readthedocs.io/en/latest/pythonapi/lightgbm.LGBMRanker.html

makes an array where each number is how many rows that belonged to that search

we need groups for both training and validation

In [ ]:
# prepare LightGBM inputs
# LightGBM ranker needs group sizes: how many hotels per search, must be sorted by srch_id first
train_sorted = train.sort_values('srch_id')
val_sorted   = val.sort_values('srch_id')

X_train = train_sorted[LGB_FEATURES]
y_train = train_sorted['relevance']
groups_train = train_sorted.groupby('srch_id').size().values

X_val   = val_sorted[LGB_FEATURES]
y_val   = val_sorted['relevance']
groups_val = val_sorted.groupby('srch_id').size().values

# train mode
model_lgb = lgb.LGBMRanker(
    objective     = 'lambdarank',
    metric        = 'ndcg',
    n_estimators  = 200,
    learning_rate = 0.05,
    num_leaves    = 31,
    # max_position  = 5,         # focus optimisation on top 5 positions
    label_gain    = [0, 1, 5], # match relevance values 0/1/5
    random_state  = 42,
    # n_jobs        = -1,
)

model_lgb.fit(
    X_train, y_train,
    group              = groups_train,
    eval_set           = [(X_val, y_val)],
    eval_group         = [groups_val],
    #eval_at            = [5],
    #callbacks          = [
    # lgb.early_stopping(stopping_rounds=100, verbose=True),
    #    lgb.log_evaluation(period=100),],
)

In [ ]:
# evaluate on validation set
val_sorted['lgb_score'] = model_lgb.predict(X_val)
lgb_ndcg = eval_ndcg5(val_sorted, score_col='lgb_score')
print(f"LightGBM NDCG@5 (validation): {lgb_ndcg:.4f}")

In [ ]:
# feature importance
importance = (pd.DataFrame({
    'feature':    LGB_FEATURES,
    'importance': model_lgb.feature_importances_,
}).sort_values('importance', ascending=False))

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(importance['feature'].head(20)[::-1],
        importance['importance'].head(20)[::-1],
        color=BLUE, edgecolor='white')
ax.set_xlabel('Feature importance (gain)')
ax.set_title('LightGBM — Top 20 Features')
plt.tight_layout()
plt.savefig('../figures/lgb_feature_importance.png', dpi=150)
plt.show()

In [ ]:
# #compare models
results = pd.DataFrame({
    'Model':   ['KNN (content-based)', 'LightGBM LambdaMART'],
    'NDCG@5':  [knn_ndcg, lgb_ndcg],
    'Type':    ['Pointwise', 'Listwise'],
})
print(results.to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(results['Model'], results['NDCG@5'],
              color=[TEAL, BLUE], edgecolor='white', width=0.5)
ax.bar_label(bars, fmt='%.4f', padding=3, fontsize=10)
ax.set_ylabel('NDCG@5')
ax.set_title('Validation NDCG@5 by Model')
ax.set_ylim(0, results['NDCG@5'].max() * 1.15)
plt.tight_layout()
plt.savefig('../figures/model_comparison.png', dpi=150)
plt.show()

In [ ]:
# evaluate on test set and save submission
test_sorted = test.sort_values('srch_id')
test_sorted['lgb_score'] = model_lgb.predict(test_sorted[LGB_FEATURES])

submission_lgb = (test_sorted
                  .sort_values(['srch_id', 'lgb_score'],
                               ascending=[True, False])
                  [['srch_id', 'prop_id']]
                  .rename(columns={'srch_id': 'SearchId',
                                   'prop_id': 'PropertyId'}))

submission_lgb.to_csv(
    f'../output/VU-DM-2026-Group-62-lgb-{timestr}.csv', index=False)
print(f"LightGBM submission saved ({len(submission_lgb):,} rows)")

In [ ]:
# Save LightGBM model
date_str = datetime.now().strftime("%Y%m%d")
joblib.dump(model_lgb, f'../models/lgb_{date_str}.pkl')
print(f"LightGBM saved: ../models/lgb_{date_str}.pkl")

### Notes

## Imputation methods
Revised  based on what each variable's missingness actually means according to the dataset description:

| Variable | Old approach | New approach | Reason |
|---|---|---|---|
| `visitor_hist_starrating`, `visitor_hist_adr_usd` | 10% trimmed mean | Fill 0 + `new_customer` flag | Null means anonymous user, not a random gap — from dataset description |
| `prop_review_score` | Median | Fill 0 + `has_review_score` flag | Description says 0 = confirmed no reviews, null = unknown — different things |
| `prop_location_score2` | Fill 0 | Destination-level median, fallback 0 | Hotels in same destination share location context |
| `orig_destination_distance` | Median | Destination-level median, fallback global median | Distance depends on destination, not random |
| `srch_query_affinity_score` | Fill 0 (already correct) | Fill 0 + `has_query_affinity` flag | Null means hotel not in internet searches |
| `gross_bookings_usd` | Dropped (already correct) | Dropped | Only present when booked — direct leakage |
| `position` | Used as feature | Never used as feature | Not in test set, encodes position bias directly |

### Feature engineering — new features added

**Within-search relative features** — *from the 5th place paper (Liu et al., arXiv:1311.7679) and the Kaggle forum top comment*
- `price_rank`, `price_ratio`, `price_rank_pct` — how cheap is this hotel relative to others shown in the same search
- `starrating_rank`, `review_rank`, `loc1_rank`, `loc2_rank` — quality ranks within the search
- These are the single most impactful feature class according to all competition solutions

**Hotel historical performance** — *from the Kaggle forum top comment and https://github.com/DavideBrex/Expedia-Hotels-recommendation-system/blob/master/feature_engineering.py*
- `prop_booking_rate`, `prop_click_rate` — how often this hotel gets booked/clicked historically
- `prop_mean_price`, `prop_std_price`, `prop_median_price` — price distribution per hotel
- `prop_n_appearances` — how many times this hotel appeared in searches
- Computed from training only to avoid leakage; price stats computed on train+test (no target involved)

**Prop-destination booking rate** — *specifically mentioned in the Kaggle forum as "worked really well"*
- `prop_dest_booking_rate`, `prop_dest_click_rate` — how this specific hotel performs at this specific destination
- Captures that the same hotel can perform very differently across destinations

**Estimated position** — *from [Igor's 2019 VU solution](https://github.com/igorpejic/personalize_expedia_hotel_searches_2013/blob/master/run.py) (4th place)*
- Average historical position per hotel+destination combination from non-random searches
- Inverted so higher = better (1/mean_position)
- Filled with 0 for unseen combos
- This is the most novel feature — it captures "how does Expedia usually rank this hotel here" without directly using position as a feature

**Quality and interaction features** — *from Davide's git solution and the 5th place paper*
- `quality_score` = star rating × review score
- `ad_vs_real` = star rating − review score (gap between advertised and actual quality)
- `bool_same_country` — user and hotel in same country
- `is_last_minute`, `is_long_stay` — search context flags
- `location_score_combined` = weighted average of both location scores

**User-hotel match features**
- `star_vs_hist` — hotel stars minus user's historical star preference
- `price_vs_hist` — hotel price minus user's historical spend

### Train/val split 
!! Need to check whether this is correct also for feature engineering


### Models

**KNN:** Feature set expanded to include the new engineered features. The NDCG scorer in CV was broken (was computing a multi-class confusion matrix, not actual NDCG) — replaced with `average_precision` which is a valid ranking proxy. Refitting after CV now uses only the training split, not the full dataset including validation.

**LightGBM:** Three fixes applied:
- Relevance target corrected from `booking_bool * 5 + click_bool` (gives 6 for booked hotels) to `click_bool + 4 * booking_bool` (correctly gives 0/1/5) !! CHECK
- Added some hyperparameters but !! CHECK 